In [ ]:
# SNP array的数据进行可视化

## read PED data from PLINK


In [ ]:
import pandas as pd
import polars as pl
import os
import sys
import numpy as np
from pathlib import Path

In [27]:
data_root = r"D:\03.projects\AI.PGT\data\pengjingjing\SNP_result\pengjingjign_203368710006"
ped_file = Path(os.path.join(data_root, "PLINK_241025_0935", "pengjingjign_203368710006.ped"))
map_file = Path(os.path.join(data_root, "PLINK_241025_0935", "pengjingjign_203368710006.map"))

ped_df = pd.read_csv(ped_file, sep="\t", header=None)
map_df = pd.read_csv(map_file, sep="\t", header=None)

# show
# print(ped_df.iloc[:, :10].head())



C:\Users\kuisu\AppData\Local\Temp\ipykernel_39404\2703771030.py:5: DtypeWarning: Columns (1896,1897,1898,1899,1902,1903,1908,1909,1914,1915,1922,1923,1924,1925,1926,1927,1928,1929,1936,1937,1938,1939,1942,1943,1944,1945,1946,1947,1948,1949,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1968,1969,1974,1975,1976,1977,1978,1979,1984,1985,1986,1987,1990,1991,1992,1993,1994,1995,1996,1997,2002,2003,2004,2005,2012,2013,2014,2015,2018,2019,2020,2021,2024,2025,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2042,2043,2044,2045,2050,2051,2058,2059,2060,2061,2074,2075,2076,2077,2078,2079,2084,2085,2086,2087,2088,2089,2092,2093,2094,2095,2098,2099,2104,2105,2106,2107,2112,2113,2114,2115,2118,2119,2130,2131,2132,2133,2134,2135,2136,2137,2138,2139,2140,2141,2150,2151,2156,2157,2170,2171,2174,2175,2176,2177,2182,2183,2186,2187,2188,2189,2190,2191,2192,2193,2202,2203,2204,2205,2218,2219,2224,2225,2226,2227,2234,2235,2238,2239,2250,2251,2252,2253,2262,2263,2266,2267,2272,2273,

In [33]:
# map file
print(map_df.head())
print(f"map file shape: {map_df.shape}")

    0            1          2          3
0  13  cnvi0111185   94.29845  100631779
1   2  cnvi0111186  183.42660  177058958
2  17  cnvi0111187   65.07668   35295593
3   8  cnvi0111188   76.01000   65499402
4   2  cnvi0111189  183.42960  177061178
map file shape: (298563, 4)


In [34]:
# ped file
# 298563 SNPs, 每个SNP两个基因型 + 5列前缀信息
print(pd.concat([ped_df.iloc[:, :5], ped_df.iloc[:, -5:]], axis=1).head())
print(f"ped file shape: {ped_df.shape}")

   0                    1       2       3       4      597127 597128 597129  \
0       1  203368710006_R01C01       0       0       0      C      G      G   
1       2  203368710006_R01C02       0       0       0      C      G      G   
2       3  203368710006_R02C01       0       0       0      C      G      G   
3       4  203368710006_R02C02       0       0       0      C      G      G   
4       5  203368710006_R03C01       0       0       0      C      G      G   

  597130 597131  
0      A      A  
1      A      A  
2      A      A  
3      A      A  
4      A      A  
ped file shape: (12, 597132)


In [35]:
print(298563 * 2 + 5)

597131


## SNP Array Data
- 读取Experiment表格
- 逐行读取，将其拼接为一个表格
- 按group的方式进行，使用dict方式按照实验组进行存储

In [36]:
import pandas as pd
import os


In [85]:
data_path=r"D:\03.projects\AI.PGT\snparray_analysis\data\experiment_snparray.csv"

with open(data_path, 'r') as file:
    lines = file.readlines()

experiments = []
experiment_dict = {}
chip_line = -1
for line in lines:
    # get experiment time
    line_data = [i.strip() for i in line.strip().split(",")]
    if "实验时间" in line:
        chip_line = 0
        if experiment_dict:
            experiments.append(experiment_dict)
            experiment_dict = {}
        time_key, time_value = line_data[1].split("：")
        experiment_dict[time_key.strip()] = time_value.strip().replace(" ", "")
        experimenter_key, experimenter_value = line_data[7].split("：")
        experiment_dict[experimenter_key.strip()] = experimenter_value.strip()
    elif "备注" in line:
        comment_key, comment_value = line_data[0][:2].strip(), line_data[0][3:].strip()
        experiment_dict[comment_key.strip()] = comment_value.strip()
        chip_line += 1

    if chip_line in [0, 3]:
        chip_line += 1
        continue

    if chip_line >= 1 and chip_line <= 5:
        experiment_dict[f"chip_num_{chip_line}"] = line_data
        chip_line += 1
    elif chip_line == 6:
        X1_key, X1_value = line_data[1].split("：")
        experiment_dict[X1_key.strip()] = X1_value.strip().replace(" ", "")
        X2_key, X2_value = line_data[7].split("：")
        experiment_dict[X2_key.strip()] = X2_value.strip()
        chip_line += 1
    elif chip_line == 7:
        X3_key, X3_value = line_data[1].split("：")
        experiment_dict[X3_key.strip()] = X3_value.strip().replace(" ", "")
        X4_key, X4_value = line_data[7].split("：")
        experiment_dict[X4_key.strip()] = X4_value.strip()
        chip_line += 1

print(len(experiments))
print(experiments[0])


9
{'实验时间': '2019年3月28日', '数据分析': '郭婧', 'chip_num_1': ['Cyto-12', 'F-徐锦荣', '柯奕K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'K8', 'K9', '梁健美L2', 'RB', ''], 'chip_num_2': ['Cyto-12', 'M-罗丽贞', 'L3', '谭立云T1', 'T2', 'T3', 'T4', 'T5', 'T6', '王锦玲W1', 'W2', 'W3', 'RB', ''], 'chip_num_4': ['Cyto-12', 'R-徐耀权', 'W4', 'W5', 'W6', 'W7', '周玲丹Z1', 'Z2', 'Z3', 'Z4', 'Z5', 'Z6', 'RB', ''], 'chip_num_5': ['Karyomap', '甘小丽G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9', 'G10', 'G11', 'G12', ''], 'X1 number': '203229580013', 'X2 number': '203229580014', 'X3 number': '203229580073', 'X4 number': '203189590151', '备注': '红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特殊备注则为同一家系外周血：F表示父亲，M-母亲，R-先证者'}


In [112]:
# 需要将chip_num_1 ~ chip_num_5 与芯片名匹配
# 需要将F-徐锦荣，柯奕K1， K2, ... -> [徐锦荣，柯奕, 柯奕, ...], [F, K1, K2, ...]
import re

class ChipInfoParser:
    def __init__(self, experiments):
        self.experiments = experiments
        self.chip_idx_dict = {"chip_num_1":"X1 number", "chip_num_2":"X2 number", "chip_num_4":"X3 number", "chip_num_5":"X4 number"}
        self.chip_info = {}
        self.name_dict ={}

    def parse_name(self, text):
        pattern = r'^([\u4e00-\u9fa5]+)([A-Z])?(\d{1,2})$'
        match = re.match(pattern, text)
        
        if match:
            chinese = match.group(1)
            initial = match.group(2) if match.group(2) else ''
            number = match.group(3)
            
            # 验证数字范围在1-20之间
            if 1 <= int(number) <= 20:
                return chinese, initial, number
    
        return None

    def parse_number(self, text):
        pattern = r'^([A-Z])(\d{1,2})$'
        match = re.match(pattern, text)
        
        if match:
            initial = match.group(1)
            number = match.group(2) if match.group(2) else ''
            
            # 验证数字范围在1-20之间
            if 2 <= int(number) <= 20:
                return initial, number
        
        return None

    def parse(self, experiments:dict):
        chip_infos = []
        name_dict ={}
        experiment_info = {}
        for key, value in experiments.items():
            chip_info = {}
            if key  in self.chip_idx_dict.keys():
                chip_info["type"] = value[0]
                chip_info["sample"] = value[1:-1]

                names = []
                name_numbers = []
                for name in chip_info["sample"]:
                    if "-" in name:
                        person_name = name.split("-")[1]
                        id_prefix = name.split("-")[0]
                        names.append(person_name)
                        name_numbers.append(id_prefix)
                    elif self.parse_name(name):
                        chinese, initial, number = self.parse_name(name)
                        name_dict[initial] = chinese
                        names.append(chinese)
                        name_numbers.append(number)
                    elif self.parse_number(name):
                        initial, number = self.parse_number(name)
                        names.append(name_dict[initial])
                        name_numbers.append(number)
                    elif "RB" == name:
                        person_name = "RB"
                        id_prefix = "1"
                        names.append(person_name)
                        name_numbers.append(id_prefix)
                    else:
                        raise ValueError(f"无法解析的名字: {name}")

                    if person_name not in name_dict:
                        name_dict[person_name] = []

                    name_dict[person_name].append(id_prefix)
                
                chip_info["names"] = names
                chip_info["name_numbers"] = name_numbers

                if self.chip_idx_dict[key] in experiments:
                    chip_info["idx"] = experiments[self.chip_idx_dict[key]]
                else:
                    raise ValueError(f"缺少芯片编号信息: {self.chip_idx_dict[key]}")
                
                if len(chip_info["names"]) == len(chip_info["name_numbers"]) == 12:
                    chip_infos.append(chip_info)
                else:
                    raise ValueError(f"样本数量不匹配 or !=12: names({len(chip_info['names'])}), name_numbers({len(chip_info['name_numbers'])})")
            
            elif key in self.chip_idx_dict.values():
                continue
            else:
                experiment_info[key] = value

        for key, value in experiment_info.items():
            for chip_info in chip_infos:
                chip_info[key] = value

        return chip_infos
    
parser = ChipInfoParser(experiments[0])
chip_infos = parser.parse(experiments[0])
for info in chip_infos:
    print(info)

{'type': 'Cyto-12', 'sample': ['F-徐锦荣', '柯奕K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'K8', 'K9', '梁健美L2', 'RB'], 'names': ['徐锦荣', '柯奕', '柯奕', '柯奕', '柯奕', '柯奕', '柯奕', '柯奕', '柯奕', '柯奕', '梁健美', 'RB'], 'name_numbers': ['F', '1', '2', '3', '4', '5', '6', '7', '8', '9', '2', '1'], 'idx': '203229580013', '实验时间': '2019年3月28日', '数据分析': '郭婧', '备注': '红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特殊备注则为同一家系外周血：F表示父亲，M-母亲，R-先证者'}
{'type': 'Cyto-12', 'sample': ['M-罗丽贞', 'L3', '谭立云T1', 'T2', 'T3', 'T4', 'T5', 'T6', '王锦玲W1', 'W2', 'W3', 'RB'], 'names': ['罗丽贞', '梁健美', '谭立云', '谭立云', '谭立云', '谭立云', '谭立云', '谭立云', '王锦玲', '王锦玲', '王锦玲', 'RB'], 'name_numbers': ['M', '3', '1', '2', '3', '4', '5', '6', '1', '2', '3', '1'], 'idx': '203229580014', '实验时间': '2019年3月28日', '数据分析': '郭婧', '备注': '红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特殊备注则为同一家系外周血：F表示父亲，M-母亲，R-先证者'}
{'type': 'Cyto-12', 'sample': ['R-徐耀权', 'W4', 'W5', 'W6', 'W7', '周玲丹Z1', 'Z2', 'Z3', 'Z4', 'Z5', 'Z6', 'RB'], 'names': ['徐耀权', '王锦玲', '王锦玲', '王锦玲', '王锦玲

In [100]:
experiments

[{'实验时间': '2019年3月28日',
  '数据分析': '郭婧',
  'chip_num_1': ['Cyto-12',
   'F-徐锦荣',
   '柯奕K1',
   'K2',
   'K3',
   'K4',
   'K5',
   'K6',
   'K7',
   'K8',
   'K9',
   '梁健美L2',
   'RB',
   ''],
  'chip_num_2': ['Cyto-12',
   'M-罗丽贞',
   'L3',
   '谭立云T1',
   'T2',
   'T3',
   'T4',
   'T5',
   'T6',
   '王锦玲W1',
   'W2',
   'W3',
   'RB',
   ''],
  'chip_num_4': ['Cyto-12',
   'R-徐耀权',
   'W4',
   'W5',
   'W6',
   'W7',
   '周玲丹Z1',
   'Z2',
   'Z3',
   'Z4',
   'Z5',
   'Z6',
   'RB',
   ''],
  'chip_num_5': ['Karyomap',
   '甘小丽G1',
   'G2',
   'G3',
   'G4',
   'G5',
   'G6',
   'G7',
   'G8',
   'G9',
   'G10',
   'G11',
   'G12',
   ''],
  'X1 number': '203229580013',
  'X2 number': '203229580014',
  'X3 number': '203229580073',
  'X4 number': '203189590151',
  '备注': '红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特殊备注则为同一家系外周血：F表示父亲，M-母亲，R-先证者'},
 {'实验时间': '2019年4月1日',
  '数据分析': '潘家富',
  'chip_num_1': ['Cyto-12',
   'M-谢彩秀',
   '谢彩秀X1',
   'X2',
   'X3',
   'X4',
   'X5',
   'X6',
   'X

In [114]:
import re

def parse_name(text):
    # 匹配中文（可能包含英文字母），然后大写字母，最后数字1-20
    pattern = r'^([\u4e00-\u9fa5A-Z]+)\s*([A-Z])\s*(\d{1,2})$'
    match = re.match(pattern, text)
    
    if match:
        chinese_part = match.group(1)
        initial = match.group(2)
        number = match.group(3)
        
        # 验证数字范围在1-20之间
        if 1 <= int(number) <= 20:
            return chinese_part, initial, number
    
    return None

# 测试
test_cases = [
    "柯奕K1", 
    "李笑玲B  L1", 
    "张三A5", 
    "王五B15", 
    "赵六20", 
    "测试C25",
    "无效输入"
]

for case in test_cases:
    result = parse_name(case)
    if result:
        print(f"'{case}' -> {result}")
    else:
        print(f"'{case}' -> 不匹配")

'柯奕K1' -> ('柯奕', 'K', '1')
'李笑玲B  L1' -> ('李笑玲B', 'L', '1')
'张三A5' -> ('张三', 'A', '5')
'王五B15' -> ('王五', 'B', '15')
'赵六20' -> 不匹配
'测试C25' -> 不匹配
'无效输入' -> 不匹配


## 读取非结构化的JSON数据


In [118]:
import pandas as pd
import json

json_path=r"D:\03.projects\AI.PGT\snparray_analysis\data\experiment_snparray.json"

with open(json_path, 'r', encoding='utf-8') as f:
    experiments = json.load(f)

# 查看数据
experiments[0]
df_experiments = pd.DataFrame(experiments)
df_experiments.head()


,type,sample,names,name_numbers,idx,实验时间,数据分析,备注
0,Cyto-12,"[F-徐锦荣, 柯奕K1, K2, K3, K4, K5, K6, K7, K8, K9, ...","[徐锦荣, 柯奕, 柯奕, 柯奕, 柯奕, 柯奕, 柯奕, 柯奕, 柯奕, 柯奕, 梁健美,...","[F, 1, 2, 3, 4, 5, 6, 7, 8, 9, 2, 1]",203229580013,2019年3月28日,郭婧,红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特...
1,Cyto-12,"[M-罗丽贞, L3, 谭立云T1, T2, T3, T4, T5, T6, 王锦玲W1, ...","[罗丽贞, 梁健美, 谭立云, 谭立云, 谭立云, 谭立云, 谭立云, 谭立云, 王锦玲, ...","[M, 3, 1, 2, 3, 4, 5, 6, 1, 2, 3, 1]",203229580014,2019年3月28日,郭婧,红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特...
2,Cyto-12,"[R-徐耀权, W4, W5, W6, W7, 周玲丹Z1, Z2, Z3, Z4, Z5,...","[徐耀权, 王锦玲, 王锦玲, 王锦玲, 王锦玲, 周玲丹, 周玲丹, 周玲丹, 周玲丹, ...","[R, 4, 5, 6, 7, 1, 2, 3, 4, 5, 6, 1]",203229580073,2019年3月28日,郭婧,红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特...
3,Karyomap,"[甘小丽G1, G2, G3, G4, G5, G6, G7, G8, G9, G10, G...","[甘小丽, 甘小丽, 甘小丽, 甘小丽, 甘小丽, 甘小丽, 甘小丽, 甘小丽, 甘小丽, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",203189590151,2019年3月28日,郭婧,红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特...
4,Cyto-12,"[M-谢彩秀, 谢彩秀X1, X2, X3, X4, X5, X6, X7, X8, X9,...","[谢彩秀, 谢彩秀, 谢彩秀, 谢彩秀, 谢彩秀, 谢彩秀, 谢彩秀, 谢彩秀, 谢彩秀, ...","[M, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1]",203229580045,2019年4月1日,潘家富,红色加粗字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无特...


In [124]:
# 需要将该表格的chipid和临床样本进行匹配
clinical_path = r"D:\03.projects\AI.PGT\snparray_analysis\data\clinical_data_demo.csv"
df_clinical = pd.read_csv(clinical_path, sep=",")
# df_clinical.head()
df_clinical[["女方姓名","病历号", "PGD/PGS编号", "活检胚胎总数"]].head()

,女方姓名,病历号,PGD/PGS编号,活检胚胎总数
0,彭晶晶,37483,1,12
1,彭晶晶,37483,2,12
2,彭晶晶,37483,3,12
3,彭晶晶,37483,4,12
4,彭晶晶,37483,9,12


In [ ]:
# 读取多个excel文件，并合并
import pandas as pd

excel_path = r"D:\01.data\01.gene\ChipData\experiment_record\5.芯片实验记录表2020.12.29（勿删）.xlsx"
df_list = pd.read_excel(excel_path, sheet_name=None, header=None)
print(df_list.keys())

Index(['type', 'idx', '实验时间', '数据分析', 'chip_sub_idx', '女方姓名', 'PGD/PGS编号',
       'sample'],
      dtype='object')


In [ ]:
df = pd.concat(df_list.values(), ignore_index=True)
df
# df_list.values()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,芯片实验记录表,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,实验时间： 2019 年 3 月 28 日,NaN,NaN,NaN,NaN,NaN,数据分析：郭婧,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Cyto-12,F-徐锦荣,柯奕K1,K2,K3,K4,K5,K6,K7,K8,K9,梁健美L2,RB,NaN,NaN
3,Cyto-12,M-罗丽贞,L3,谭立云T1,T2,T3,T4,T5,T6,王锦玲W1,W2,W3,RB,NaN,NaN
4,NaN,R01C01,R02C01,R03C01,R04C01,R05C01,R06C01,R01C02,R02C02,R03C02,R04C02,R05C02,R06C02,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2545,Karyomap,F-方慧,M-周海兰(预实验),R-吕桂玉,F-魏志诚,M-曾丽兰(预实验),R-魏伊依,丘兰兰Q1,Q2,Q3,Q4,Q5,周诗敏Z1,NaN,NaN
2546,NaN,X1 number：204289520017 ...,NaN,NaN,NaN,NaN,NaN,X2 number：204270360032,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2547,NaN,X3 number：204270360069 ...,NaN,NaN,NaN,NaN,NaN,X4 number：204694750147,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2548,备注：红色字体表示需要区分正常和携带；Cyto-12芯片中R01C01为阳性对照外周血，如无...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
### 读取experiment和clinical的数据表，将snp id和clinical id合并

import pandas as pd

snparray_path = r"D:\03.projects\AI.PGT\snparray_analysis\data\5.芯片实验记录表2020.12.29_persons.csv"
clinical_path = r"D:\03.projects\AI.PGT\snparray_analysis\data\clinical_data_pgt.csv"
df_snparray = pd.read_csv(snparray_path,header=0)
print(df_snparray.head())

df_clinical = pd.read_csv(clinical_path,header=1)
print(df_clinical.head())

# 创建一个新的列，将女方姓名和PGD/PGS编号合并 彭晶晶-1


      type           idx        实验时间 数据分析 chip_sub_idx 女方姓名 PGD/PGS编号 sample
0  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R01C01  徐锦荣         F  F-徐锦荣
1  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R02C01   柯奕         1   柯奕K1
2  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R03C01   柯奕         2     K2
3  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R04C01   柯奕         3     K3
4  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R05C01   柯奕         4     K4
     病历号       统计日期      周期号 女方姓名  女方年龄 男方姓名  男方年龄  第几周期   周期类型          助孕方案  \
0  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   
1  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   
2  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   
3  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   
4  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   

   ... D6滋养层 D7分期 D7内细胞团 D7滋养层 诊断不可移植 是否推荐移植       

In [ ]:
# add new col
df_snparray['女方姓名-PGD/PGS编号'] = df_snparray.apply(lambda row: f"{row['女方姓名']}-{row['PGD/PGS编号']}", axis=1)
print(df_snparray.head())

df_clinical['女方姓名-PGD/PGS编号'] = df_clinical.apply(lambda row: f"{row['女方姓名']}-{row['PGD/PGS编号']}", axis=1)
print(df_clinical.head())

      type           idx        实验时间 数据分析 chip_sub_idx 女方姓名 PGD/PGS编号 sample  \
0  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R01C01  徐锦荣         F  F-徐锦荣   
1  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R02C01   柯奕         1   柯奕K1   
2  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R03C01   柯奕         2     K2   
3  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R04C01   柯奕         3     K3   
4  Cyto-12  2.032296e+11  2019年3月28日   郭婧       R05C01   柯奕         4     K4   

  女方姓名-PGD/PGS编号  
0          徐锦荣-F  
1           柯奕-1  
2           柯奕-2  
3           柯奕-3  
4           柯奕-4  
     病历号       统计日期      周期号 女方姓名  女方年龄 男方姓名  男方年龄  第几周期   周期类型          助孕方案  \
0  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   
1  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   
2  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   
3  37483  2019/6/11  1057662  彭晶晶    25   李勇    28     2  新鲜加解冻  PGT_A+PGT_SR   


In [12]:
# merge 
merged_inner = pd.merge(df_clinical, df_snparray, on='女方姓名-PGD/PGS编号', how='inner')
print(merged_inner.head())
print(merged_inner.shape)

     病历号       统计日期      周期号 女方姓名_x  女方年龄 男方姓名  男方年龄  第几周期   周期类型  \
0  37483  2019/6/11  1057662    彭晶晶    25   李勇    28     2  新鲜加解冻   
1  37483  2019/6/11  1057662    彭晶晶    25   李勇    28     2  新鲜加解冻   
2  37483  2019/6/11  1057662    彭晶晶    25   李勇    28     2  新鲜加解冻   
3  37483  2019/6/11  1057662    彭晶晶    25   李勇    28     2  新鲜加解冻   
4  37483  2019/6/11  1057662    彭晶晶    25   李勇    28     2  新鲜加解冻   

           助孕方案  ...  E_IDNo 女方姓名-PGD/PGS编号     type           idx  \
0  PGT_A+PGT_SR  ...  554274          彭晶晶-1  Cyto-12  2.033687e+11   
1  PGT_A+PGT_SR  ...  554278          彭晶晶-2  Cyto-12  2.033687e+11   
2  PGT_A+PGT_SR  ...  554279          彭晶晶-3  Cyto-12  2.033687e+11   
3  PGT_A+PGT_SR  ...  554280          彭晶晶-4  Cyto-12  2.033687e+11   
4  PGT_A+PGT_SR  ...  554281          彭晶晶-9  Cyto-12  2.033687e+11   

         实验时间 数据分析 chip_sub_idx 女方姓名_y PGD/PGS编号_y sample  
0  2019年6月24日   李荣       R02C01    彭晶晶           1  彭晶晶P1  
1  2019年6月24日   李荣       R03C01    彭晶晶      